# C-1N: your first policy

Edit a policy, record its motion, and compare measurements with its replay.
The code path is **this notebook → `c1n/learning.py` → `c1n/simulation.py`**.
The helper owns recording and display. You own the policy, observation, reward,
action timing, and later RL implementation.

Run the setup cell below. It loads the model and resets it; it does not advance physics.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "spider" / "learning.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel in the repository or lab/notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from spider.learning import LearningSimulation

sim = LearningSimulation()
measured = sim.reset()
print("Number of actuator targets:", sim.model.nu)
from spider.learning import record_policy, plot_recordings

print("Physics timestep (s):", sim.model.opt.timestep)
print("Measurement fields:", ", ".join(measured.__dataclass_fields__))
print("Actuators:", [(i, sim.model.actuator(i).name) for i in range(sim.model.nu)])


## Edit here: compare corresponding joints on two legs

The control preserves the two front-left offsets and restores a scalar comparison with your current 0.5-radian threshold. The failed all-joint draft is saved locally in `telemetry/paired-leg-comparison/before.ipynb`.

Actuators 0, 1, 2 are front-left coxa, hip, knee; 3, 4, 5 are the corresponding front-right joints. `right_sign=0` leaves the right leg neutral, `+1` copies the offsets, and `-1` reverses them. These are offsets from neutral, so opposite offsets do not necessarily produce opposite absolute angles after clipping.

Compare the three fixed-reset trials below. No training or loss update occurs.

In [ ]:
def policy(observation, right_sign=0):
    offsets = np.zeros(18, dtype=float)
    offsets[0] = 1
    offsets[1] = -1

    measured = observation.joint_positions[0]
    desired = 0.5  # radians; your current comparison threshold
    if measured > desired:
        offsets[0] = -1

    # Copy this leg's commands to the corresponding joints on the other leg.
    offsets[3:6] = right_sign * offsets[0:3]
    return offsets

In [ ]:
action = np.asarray(policy(measured), dtype=float)
assert action.shape == (sim.model.nu,)
assert np.isfinite(action).all()
print(action)

## Run the bounded comparison

Each treatment resets the robot, runs 100 actions at 0.02 seconds per action,
and records the same measurements and replay states. Change only `right_sign`
between treatments. The left leg still supplies the feedback measurement in all
three cases; the right leg does not have a separate feedback controller.

The table reports motion and support, not a loss or a claim of useful walking.
Saved traces and models live in `telemetry/paired-leg-comparison/`.

In [ ]:
import mujoco
import pandas as pd
from functools import partial

physics_steps = 10
action_count = 100
cases = {"left only": 0, "same signs": 1, "opposite signs": -1}
recordings = {}
rows = []
output_dir = REPO_ROOT / "telemetry" / "paired-leg-comparison"
output_dir.mkdir(parents=True, exist_ok=True)

for label, right_sign in cases.items():
    recording = record_policy(partial(policy, right_sign=right_sign), label=label,
                              physics_steps=physics_steps, action_count=action_count)
    recordings[label] = recording
    states = recording.measurements
    xyz = np.asarray([state.torso_position for state in states])
    contacts = np.asarray([len(state.foot_contacts) for state in states])
    times = np.asarray(recording.replay.times)
    expected_times = np.arange(action_count + 1) * physics_steps * recording.replay.model.opt.timestep
    valid_timing = bool(np.allclose(times, expected_times, rtol=0, atol=1e-9))
    rows.append({"treatment": label,
                 "valid_timing": valid_timing,
                 "final_sim_time_s": times[-1],
                 "dx_m": xyz[-1, 0] - xyz[0, 0],
                 "dy_m": xyz[-1, 1] - xyz[0, 1],
                 "final_height_m": xyz[-1, 2],
                 "minimum_contacts": int(contacts.min())})
    run_dir = output_dir / label.replace(" ", "-")
    run_dir.mkdir(exist_ok=True)
    mujoco.mj_saveModel(recording.replay.model, str(run_dir / "model.mjb"))
    np.savez(run_dir / "states.npz", states=np.asarray(recording.replay.states),
             times=np.asarray(recording.replay.times), label=recording.replay.label,
             actuator_name="front_right_coxa_motor")
    np.savez(run_dir / "measurements.npz", times=np.asarray(recording.replay.times),
             torso_position=xyz, contacts=contacts, offsets=recording.offsets,
             targets=recording.targets, action_times=recording.action_times)

results = pd.DataFrame(rows).set_index("treatment")
display(results)
results.to_csv(output_dir / "summary.csv")
fig, axes = plot_recordings(*recordings.values(), actuator=3)
fig.savefig(output_dir / "comparison.png", dpi=150)

### Recorded failure in this comparison

The **same signs** run emitted a MuJoCo QACC instability warning at 1.982 s.
Its simulation clock reset and ended at 0.018 s instead of 2 s. Its final displacement
is therefore **not a valid two-second locomotion comparison**. The original plot
and exact replay retain that failure; the jump to the reset pose is not a gait cycle.

The other two runs reached 2 s with monotonic timestamps. All three sampled zero
foot contacts at some point. Numerical timing validity does not establish stable
or useful locomotion. No gains or amplitudes were changed to hide the failure.

## Inspect the recorded movement

Choose one of the three labels below to replay its exact recorded states.
The plot compares all three trials; the selected viewer shows one treatment.
Look for changes in body direction, height and foot contact before explaining why they occurred.

In [ ]:
replay_case = "same signs"  # "left only", "same signs", "opposite signs"
viewer = recordings[replay_case].watch(REPO_ROOT / "telemetry" / "policy-replays", speed=1.0)

Which movement changes when the right leg joins in? Does reversing its offsets reverse the body motion, or change support in another way? These three samples test coordination; they do not demonstrate convergence or a learned gait.